# Figure: rebuild churn is local, and the gradient is exact

Loads `results/differentiability/nn_rebuild_scaling_step_scaled.json` and
`results/differentiability/nn_rebuild_gradient_check.json`
(`bench/differentiability/nn_rebuild.py`). Never recomputes.

**(a)** Why the original switch metric had to be replaced. `topology_changed` asks whether the
length-$N$ Morton permutation changed *at all* — an **extensive** indicator, driven to 1 by $N$
alone (if each particle changes slot with probability $p$, $P(\text{no change})\sim(1-p)^N$). It
saturates at 1.000 from $N=16\,384$ and says nothing thereafter. The **intensive** per-particle
rates resolve a clean monotone trend over exactly that range: the ordering churns pervasively
but *locally* — most particles that shuffle within the ordering stay in the same leaf, and the
mean displacement *falls* as $N$ grows. Series C (density-matched $r^*$, $N$-scaled step), 3
seeds, error bars are the seed standard deviation. Series A and B are the controls and are not
plotted; see `results/differentiability/README.md`.

**(b)** The gradient certificate. Differentiating *through* the per-step rebuild is bit-identical
to differentiating at a frozen ordering, at every $N$ tested. Central differences then confirm
that gradient — but only once **both** discrete choices are pinned: the tree ordering *and* the
`argmin` inside the objective's `min` over neighbour candidates. Pinning only the tree leaves a
residual that grows to $1.6\times10^{-2}$ and reads like a gradient bug; it is the argmin.
Unpinned differences straddle a rebuild boundary and are legitimately wrong.

Caveat carried by the data, not just the caption: every rate here is **permutation-derived**
(`switch_metric: "morton_ordering"`), including the leaf rate. It is *not* an interaction-switch
rate and must not be drawn beside one.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_here = Path.cwd()
_candidates = [
    _here / "results" / "differentiability",
    _here.parents[1] / "results" / "differentiability",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PALETTE = {"radix": "#4C72B0", "octree": "#DD8452", "kdtree": "#55A868"}
BASELINE = "#8C8C8C"

In [ ]:
series = json.loads((RESULTS / "nn_rebuild_scaling_step_scaled.json").read_text())
check = json.loads((RESULTS / "nn_rebuild_gradient_check.json").read_text())
meta = series["metadata"]

agg = series["aggregate"]
N = np.array([a["num_particles"] for a in agg])
any_change = np.array([a["topology_change_fraction"]["mean"] for a in agg])
slot = np.array([a["slot_change_fraction"]["mean"] for a in agg])
slot_sd = np.array([a["slot_change_fraction"]["std"] for a in agg])
leaf = np.array([a["leaf_change_fraction"]["mean"] for a in agg])
leaf_sd = np.array([a["leaf_change_fraction"]["std"] for a in agg])
rank_shift = np.array([a["mean_abs_rank_shift_normalized"]["mean"] for a in agg])
loss_ratio = np.array([a["loss_ratio"]["mean"] for a in agg])

done = [r for r in check["records"] if "error" not in r]
f64 = sorted(
    (r for r in done if r["dtype"] == "float64"), key=lambda r: r["num_particles"]
)
gN = np.array([r["num_particles"] for r in f64])
fd_full = np.array([r["best_pinned_full"]["rel_err_pinned_full"] for r in f64])
fd_tree = np.array([r["best_pinned"]["rel_err_pinned"] for r in f64])
fd_unpinned = np.array([r["best_pinned"]["rel_err_rebuilt"] for r in f64])

# The exactness claim is asserted here rather than asserted in the caption.
transparency = [r["rebuild_transparency"]["max_abs_grad_difference"] for r in done]
assert set(transparency) == {0.0}, transparency
n_exact = max(r["num_particles"] for r in done)

print(f"device={meta['device_kind']} jax={meta['jax_version']} commit={meta.get('git_commit')}")
print(f"series={series['series']}  switch_metric={series['switch_metric']}")
print(f"churn: N={N.min()}..{N.max()}  loss ratio {loss_ratio.min():.2e}..{loss_ratio.max():.2e}")
print(f"grad: exactly-zero rebuild gap at all {len(done)} runs, up to N={n_exact}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), constrained_layout=True)

# --- (a) extensive indicator vs intensive rates ---------------------------
ax = axes[0]
ax.set_xscale("log")
ax.plot(
    N, any_change, ls="--", marker="s", ms=4, color=BASELINE,
    label="any change (extensive)",
)
ax.errorbar(
    N, slot, yerr=slot_sd, marker="o", ms=4, capsize=3, color=PALETTE["radix"],
    label="slot changed",
)
ax.errorbar(
    N, leaf, yerr=leaf_sd, marker="^", ms=4, capsize=3, color=PALETTE["octree"],
    label="leaf changed",
)
ax.set_ylim(-0.04, 1.09)
ax.set_xlabel("$N$ (particles)")
ax.set_ylabel("fraction per optimizer step")
ax.set_title("(a) rebuild churn: pervasive but local")

sat = N[any_change >= 0.9995]
if sat.size:
    ax.annotate(
        "saturated: no resolution",
        xy=(sat[len(sat) // 2], 1.0), xytext=(0, -34), textcoords="offset points",
        ha="center", fontsize=9, color=BASELINE,
        arrowprops=dict(arrowstyle="->", color=BASELINE, lw=0.9),
    )
ax.text(
    0.03, 0.055,
    f"converging at every $N$:\nloss ratio {loss_ratio.min():.1e}–{loss_ratio.max():.1e}",
    transform=ax.transAxes, fontsize=9, color="#444444", va="bottom",
)

ax_r = ax.twinx()
ax_r.plot(N, rank_shift, marker="d", ms=4, color=PALETTE["kdtree"])
ax_r.set_yscale("log")
ax_r.set_ylabel(
    r"mean $|\Delta\mathrm{rank}|\,/\,N$", color=PALETTE["kdtree"]
)
ax_r.tick_params(axis="y", labelcolor=PALETTE["kdtree"])
ax_r.grid(False)

handles, labels = ax.get_legend_handles_labels()
handles.append(plt.Line2D([], [], marker="d", ms=4, color=PALETTE["kdtree"]))
labels.append(r"mean $|\Delta\mathrm{rank}|/N$ (right)")
ax.legend(handles, labels, loc="upper left", fontsize=9)

# --- (b) the gradient certificate -----------------------------------------
ax = axes[1]
ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(
    gN, fd_unpinned, marker="s", ms=4, color=BASELINE,
    label="FD unpinned (straddles a rebuild)",
)
ax.plot(
    gN, fd_tree, marker="^", ms=5, ls="--", lw=1.6, color=PALETTE["octree"],
    label="FD, tree pinned only",
)
ax.plot(
    gN, fd_full, marker="o", ms=4, color=PALETTE["radix"],
    label="FD, tree + argmin pinned",
)
ax.axhline(1e-6, color=BASELINE, lw=0.8, ls=":")
ax.text(
    gN[0], 1.35e-6, "correctness gate", fontsize=8, color=BASELINE, va="bottom"
)
ax.set_xlabel("$N$ (particles)")
ax.set_ylabel("relative error vs. autodiff")
ax.set_title("(b) gradient certificate (float64)")
ax.legend(loc="center left", fontsize=9)

n_pretty = f"{n_exact:,}".replace(",", r"\,")
exact_label = (
    r"$\max|\nabla L_{\mathrm{rebuilt}}-\nabla L_{\mathrm{pinned}}| \equiv 0$"
    "  exactly, "
    rf"$N \leq {n_pretty}$"
)
ax.text(
    0.5, 0.035, exact_label,
    transform=ax.transAxes, ha="center", va="bottom", fontsize=9,
    color=PALETTE["radix"],
    bbox=dict(boxstyle="round,pad=0.35", fc="#EEF2F8", ec=PALETTE["radix"], lw=0.8),
)

out = FIGDIR / "fig_nn_rebuild_scaling.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)